# Cam Ring Edge Detection and Dimensional Analysis

## Overview
This notebook demonstrates edge detection and dimensional analysis of a cam ring from a top-down image. The workflow includes:
- Image preprocessing and edge detection
- Center identification
- Polar coordinate conversion
- Radial distance measurement across angles
- Visualization and summary statistics

## 1. Import Required Libraries

Import necessary libraries including OpenCV, NumPy, Matplotlib, SciPy, and Pandas for image processing, numerical computation, visualization, and data analysis.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib to display inline
%matplotlib inline
print("All required libraries imported successfully!")

## 2. Load and Display the Cam Ring Image

Load the cam ring image from the provided path and display it using Matplotlib to understand the input data.

In [ ]:
# First, let's generate a synthetic cam ring image for demonstration if needed
# You can replace this with your own image

from src.image_generator import create_synthetic_cam_ring

# Path to save/load image
image_path = 'data/cam_ring_test.png'
Path('data').mkdir(exist_ok=True)

# Create a synthetic cam ring for demonstration
synthetic_image = create_synthetic_cam_ring(
    width=800,
    height=800,
    inner_radius=150,
    outer_radius=250,
    noise_level=10,
    output_path=image_path
)

# Load the image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Display the image
plt.figure(figsize=(10, 8))
plt.imshow(image_rgb)
plt.title('Loaded Cam Ring Image', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")

## 3. Preprocess the Image

Apply preprocessing techniques such as grayscale conversion, Gaussian blur, and contrast adjustment to prepare the image for edge detection.

In [ ]:
# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Apply Gaussian blur to reduce noise
blur_kernel = 5
blurred = cv2.GaussianBlur(gray, (blur_kernel, blur_kernel), 1.5)

# Apply histogram equalization to enhance contrast
equalized = cv2.equalizeHist(blurred)

# Display preprocessing steps
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Grayscale Image', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(blurred, cmap='gray')
axes[0, 1].set_title('After Gaussian Blur (kernel=5)', fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(equalized, cmap='gray')
axes[1, 0].set_title('After Histogram Equalization', fontweight='bold')
axes[1, 0].axis('off')

# Histogram comparison
axes[1, 1].hist(gray.flatten(), bins=256, color='blue', alpha=0.5, label='Original')
axes[1, 1].hist(equalized.flatten(), bins=256, color='red', alpha=0.5, label='Equalized')
axes[1, 1].set_xlabel('Pixel Value')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Histogram Comparison', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("Preprocessing completed!")

## 4. Apply Edge Detection

Use Canny edge detection to identify edges in the preprocessed image. Tune parameters to detect both inner and outer edges clearly.

In [ ]:
# Apply Canny edge detection
low_threshold = 50
high_threshold = 150

edges = cv2.Canny(equalized, low_threshold, high_threshold)

# Display edge detection results with different parameters
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Low threshold levels
canny_50_100 = cv2.Canny(equalized, 50, 100)
axes[0, 0].imshow(canny_50_100, cmap='gray')
axes[0, 0].set_title('Canny (50, 100)', fontweight='bold')
axes[0, 0].axis('off')

canny_50_150 = cv2.Canny(equalized, 50, 150)
axes[0, 1].imshow(canny_50_150, cmap='gray')
axes[0, 1].set_title('Canny (50, 150) - Selected', fontweight='bold')
axes[0, 1].axis('off')

canny_100_200 = cv2.Canny(equalized, 100, 200)
axes[1, 0].imshow(canny_100_200, cmap='gray')
axes[1, 0].set_title('Canny (100, 200)', fontweight='bold')
axes[1, 0].axis('off')

canny_30_100 = cv2.Canny(equalized, 30, 100)
axes[1, 1].imshow(canny_30_100, cmap='gray')
axes[1, 1].set_title('Canny (30, 100)', fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"Edge detection completed!")
print(f"Number of edge pixels detected: {np.sum(edges > 0)}")

## 5. Find the Center of the Cam Ring

Detect the center of the cam ring using contour analysis, moment-based methods, and circle fitting.

In [ ]:
# Find contours in the edge image
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if len(contours) > 0:
    # Find the largest contour (should be the cam ring)
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Method 1: Center of mass using moments
    M = cv2.moments(largest_contour)
    if M["m00"] != 0:
        cx_moments = int(M["m10"] / M["m00"])
        cy_moments = int(M["m01"] / M["m00"])
    else:
        h, w = edges.shape
        cx_moments, cy_moments = w // 2, h // 2
    
    # Method 2: Min enclosing circle
    (x_circle, y_circle), radius_circle = cv2.minEnclosingCircle(largest_contour)
    cx_circle = int(x_circle)
    cy_circle = int(y_circle)
    
    # Use the circle-fitting method as it's more robust
    center = (cx_circle, cy_circle)
    
    print(f"Center found using moments method: ({cx_moments}, {cy_moments})")
    print(f"Center found using circle fitting: ({cx_circle}, {cy_circle})")
    print(f"Minimal enclosing circle radius: {radius_circle:.2f}")
else:
    # Fallback to image center
    h, w = edges.shape
    center = (w // 2, h // 2)
    print(f"No contours found. Using image center: {center}")

# Visualize center detection
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Show contours
contour_image = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
cv2.drawContours(contour_image, [largest_contour], 0, (0, 255, 0), 2)
contour_image_rgb = cv2.cvtColor(contour_image, cv2.COLOR_BGR2RGB)
axes[0].imshow(contour_image_rgb)
axes[0].set_title('Detected Contours', fontweight='bold')
axes[0].axis('off')

# Show center and circle
center_image = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
cv2.circle(center_image, center, 5, (0, 0, 255), -1)  # Center in red
cv2.circle(center_image, center, int(radius_circle), (255, 0, 0), 2)  # Circle in blue
center_image_rgb = cv2.cvtColor(center_image, cv2.COLOR_BGR2RGB)
axes[1].imshow(center_image_rgb)
axes[1].set_title('Detected Center and Min Circle', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"\nCenter position: {center}")

## 6. Extract Edge Coordinates

Extract the coordinates of all detected edges from the edge map and store them for further analysis.

In [ ]:
# Extract coordinates of all edge pixels
# argwhere returns (row, col) which corresponds to (y, x)
edge_coords = np.argwhere(edges > 0)

print(f"Total edge pixels detected: {len(edge_coords)}")
print(f"Edge coordinate format: (y, x) from argwhere")
print(f"Sample edge coordinates (first 10):")
print(edge_coords[:10])

# Convert to (x, y) format for easier interpretation
edge_x = edge_coords[:, 1]
edge_y = edge_coords[:, 0]

# Display spatial distribution of edge pixels
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot
axes[0].scatter(edge_x, edge_y, c='blue', s=1, alpha=0.5)
axes[0].plot(center[0], center[1], 'r*', markersize=20, label='Center')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
axes[0].set_title('Spatial Distribution of Edge Pixels', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Distance from center histogram
distances_from_center = np.sqrt((edge_x - center[0])**2 + (edge_y - center[1])**2)
axes[1].hist(distances_from_center, bins=50, color='green', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Distance from Center (pixels)')
axes[1].set_ylabel('Number of Pixels')
axes[1].set_title('Histogram of Distances from Center', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDistance statistics from center:")
print(f"  Min: {distances_from_center.min():.2f} pixels")
print(f"  Max: {distances_from_center.max():.2f} pixels")
print(f"  Mean: {distances_from_center.mean():.2f} pixels")
print(f"  Std: {distances_from_center.std():.2f} pixels")

## 7. Convert to Polar Coordinates

Convert edge coordinates from Cartesian to polar coordinates (radius and angle) relative to the detected center using trigonometric functions.

In [ ]:
# Convert Cartesian coordinates to polar coordinates
# edge_coords contains (y, x), so we need to adjust
cx, cy = center

# Calculate differences from center
dy = edge_coords[:, 0] - cy  # y component
dx = edge_coords[:, 1] - cx  # x component

# Calculate angles in radians using atan2
angles_rad = np.arctan2(dy, dx)

# Convert to degrees and normalize to 0-360 range
angles_deg = np.degrees(angles_rad)
angles_deg = (angles_deg + 360) % 360

# Calculate radii
radii = np.sqrt(dx**2 + dy**2)

print(f"Polar coordinate conversion completed!")
print(f"Number of points: {len(angles_deg)}")
print(f"\nAngle statistics (degrees):")
print(f"  Min: {angles_deg.min():.2f}°")
print(f"  Max: {angles_deg.max():.2f}°")
print(f"  Mean: {angles_deg.mean():.2f}°")
print(f"\nRadius statistics (pixels):")
print(f"  Min: {radii.min():.2f}")
print(f"  Max: {radii.max():.2f}")
print(f"  Mean: {radii.mean():.2f}")

# Visualize polar coordinates
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot: angle vs radius
axes[0].scatter(angles_deg, radii, c='blue', s=1, alpha=0.5)
axes[0].set_xlabel('Angle (degrees)')
axes[0].set_ylabel('Radius (pixels)')
axes[0].set_title('Radius vs Angle in Polar Coordinates', fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 360)

# Polar plot
ax = plt.subplot(1, 2, 2, projection='polar')
angles_rad_plot = np.deg2rad(angles_deg)
ax.scatter(angles_rad_plot, radii, c='blue', s=1, alpha=0.5)
ax.set_title('Polar Representation of Edges', fontweight='bold', pad=20)
ax.grid(True)

plt.tight_layout()
plt.show()

## 8. Analyze Radius at Angular Intervals

Sample radius values at regular angular intervals (e.g., every 1 degree) for the inner and outer edges to create a consistent dataset.

In [ ]:
# Define angular resolution (in degrees)
angle_resolution = 1  # Analyze every 1 degree

# Create angular bins
angle_bins = np.arange(0, 360, angle_resolution)

# For each angular bin, find min and max radii
inner_radii = []
outer_radii = []
bin_centers = []

for angle_center in angle_bins:
    angle_range = angle_resolution / 2
    
    # Find all radii within this angular bin
    in_range = (angles_deg >= angle_center - angle_range) & (angles_deg < angle_center + angle_range)
    
    if np.any(in_range):
        radii_in_range = radii[in_range]
        # Inner radius is minimum, outer radius is maximum
        inner_r = np.min(radii_in_range)
        outer_r = np.max(radii_in_range)
        
        inner_radii.append(inner_r)
        outer_radii.append(outer_r)
        bin_centers.append(angle_center)

# Convert to numpy arrays
bin_centers = np.array(bin_centers)
inner_radii = np.array(inner_radii)
outer_radii = np.array(outer_radii)

print(f"Angular analysis completed!")
print(f"Number of angular bins analyzed: {len(bin_centers)}")
print(f"\nInner radius statistics:")
print(f"  Min: {inner_radii.min():.2f} pixels")
print(f"  Max: {inner_radii.max():.2f} pixels")
print(f"  Mean: {inner_radii.mean():.2f} pixels")
print(f"  Std: {inner_radii.std():.2f} pixels")
print(f"\nOuter radius statistics:")
print(f"  Min: {outer_radii.min():.2f} pixels")
print(f"  Max: {outer_radii.max():.2f} pixels")
print(f"  Mean: {outer_radii.mean():.2f} pixels")
print(f"  Std: {outer_radii.std():.2f} pixels")

# Visualize inner and outer radii
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Cartesian plot
axes[0].plot(bin_centers, inner_radii, 'b-', linewidth=2, label='Inner Radius')
axes[0].plot(bin_centers, outer_radii, 'r-', linewidth=2, label='Outer Radius')
axes[0].fill_between(bin_centers, inner_radii, outer_radii, alpha=0.3, color='green')
axes[0].set_xlabel('Angle (degrees)')
axes[0].set_ylabel('Radius (pixels)')
axes[0].set_title('Inner and Outer Radii vs Angle', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 360)

# Polar plot
ax = plt.subplot(1, 2, 2, projection='polar')
angles_rad_plot = np.deg2rad(bin_centers)
ax.plot(angles_rad_plot, inner_radii, 'b-', linewidth=2, label='Inner')
ax.plot(angles_rad_plot, outer_radii, 'r-', linewidth=2, label='Outer')
ax.fill_between(angles_rad_plot, inner_radii, outer_radii, alpha=0.3, color='green')
ax.set_title('Polar View of Radii', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

## 9. Calculate Distance Between Edges

Compute the radial distance between the inner and outer edges for each angular position to quantify the cam ring thickness.

In [ ]:
# Calculate distance between inner and outer edges
distances = outer_radii - inner_radii

print(f"Distance calculation completed!")
print(f"\nDistance statistics (pixels):")
print(f"  Min: {distances.min():.2f}")
print(f"  Max: {distances.max():.2f}")
print(f"  Mean: {distances.mean():.2f}")
print(f"  Median: {np.median(distances):.2f}")
print(f"  Std Dev: {distances.std():.2f}")

# Calculate variation metrics
max_variation = distances.max() - distances.min()
cv = (distances.std() / distances.mean()) * 100  # Coefficient of variation

print(f"\nVariation Analysis:")
print(f"  Total Variation: {max_variation:.2f} pixels")
print(f"  Coefficient of Variation: {cv:.2f}%")

# Find angular positions with min/max distance
min_distance_angle = bin_centers[np.argmin(distances)]
max_distance_angle = bin_centers[np.argmax(distances)]

print(f"\nExtreme Values:")
print(f"  Minimum distance ({distances.min():.2f} px) at angle: {min_distance_angle:.1f}°")
print(f"  Maximum distance ({distances.max():.2f} px) at angle: {max_distance_angle:.1f}°")

# Visualize distances
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Distance vs Angle (Cartesian)
axes[0, 0].plot(bin_centers, distances, 'b-', linewidth=2, marker='o', markersize=3)
axes[0, 0].axhline(distances.mean(), color='r', linestyle='--', label=f'Mean: {distances.mean():.2f}px')
axes[0, 0].fill_between(bin_centers, distances, alpha=0.3)
axes[0, 0].set_xlabel('Angle (degrees)')
axes[0, 0].set_ylabel('Distance (pixels)')
axes[0, 0].set_title('Radial Distance Between Edges vs Angle', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim(0, 360)

# Plot 2: Distance distribution (histogram)
axes[0, 1].hist(distances, bins=30, color='green', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(distances.mean(), color='r', linestyle='--', linewidth=2, label='Mean')
axes[0, 1].axvline(np.median(distances), color='orange', linestyle='--', linewidth=2, label='Median')
axes[0, 1].set_xlabel('Distance (pixels)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Distances', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Polar representation of distances
ax = plt.subplot(2, 2, 3, projection='polar')
angles_rad_plot = np.deg2rad(bin_centers)
ax.plot(angles_rad_plot, distances, 'b-', linewidth=2)
ax.fill(angles_rad_plot, distances, alpha=0.3, color='green')
ax.set_title('Polar View of Distances', fontweight='bold', pad=20)
ax.grid(True)

# Plot 4: Cumulative statistics
axes[1, 1].plot(bin_centers, np.cumsum(distances), 'purple', linewidth=2, label='Cumulative')
axes[1, 1].set_xlabel('Angle (degrees)')
axes[1, 1].set_ylabel('Cumulative Distance (pixels)')
axes[1, 1].set_title('Cumulative Distance', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim(0, 360)

plt.tight_layout()
plt.show()

## 10. Visualize Results

Create comprehensive visualizations including the original image with detected edges overlaid, polar plots of radius values, and distance plots across different angles.

In [ ]:
# Create comprehensive visualization with overlays
fig = plt.figure(figsize=(16, 12))

# 1. Original image with edges detected
ax1 = plt.subplot(2, 3, 1)
ax1.imshow(image_rgb)
ax1.set_title('Original Image', fontweight='bold')
ax1.axis('off')

# 2. Detected edges
ax2 = plt.subplot(2, 3, 2)
ax2.imshow(edges, cmap='gray')
ax2.set_title('Detected Edges (Canny)', fontweight='bold')
ax2.axis('off')

# 3. Edges with center marked
ax3 = plt.subplot(2, 3, 3)
edges_vis = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
cv2.circle(edges_vis, center, 5, (0, 0, 255), -1)
cv2.circle(edges_vis, center, int(radius_circle), (255, 0, 0), 2)
edges_vis_rgb = cv2.cvtColor(edges_vis, cv2.COLOR_BGR2RGB)
ax3.imshow(edges_vis_rgb)
ax3.set_title('Detected Center and Ring', fontweight='bold')
ax3.axis('off')

# 4. Radius distribution (Cartesian)
ax4 = plt.subplot(2, 3, 4)
ax4.plot(bin_centers, inner_radii, 'b-', linewidth=2, label='Inner')
ax4.plot(bin_centers, outer_radii, 'r-', linewidth=2, label='Outer')
ax4.fill_between(bin_centers, inner_radii, outer_radii, alpha=0.3, color='green')
ax4.set_xlabel('Angle (degrees)')
ax4.set_ylabel('Radius (pixels)')
ax4.set_title('Radius Profile', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0, 360)

# 5. Distance distribution
ax5 = plt.subplot(2, 3, 5)
ax5.plot(bin_centers, distances, 'g-', linewidth=2)
ax5.fill_between(bin_centers, distances, alpha=0.3, color='green')
ax5.axhline(distances.mean(), color='r', linestyle='--', label=f'Mean: {distances.mean():.2f}px')
ax5.set_xlabel('Angle (degrees)')
ax5.set_ylabel('Distance (pixels)')
ax5.set_title('Radial Distance Between Edges', fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)
ax5.set_xlim(0, 360)

# 6. Overlay of inner and outer edges on original
ax6 = plt.subplot(2, 3, 6)
overlay = image_rgb.copy()
for i, angle in enumerate(bin_centers):
    if i % 5 == 0:  # Draw every 5th ray to avoid clutter
        angle_rad = np.deg2rad(angle)
        x_inner = int(center[0] + inner_radii[i] * np.cos(angle_rad))
        y_inner = int(center[1] + inner_radii[i] * np.sin(angle_rad))
        x_outer = int(center[0] + outer_radii[i] * np.cos(angle_rad))
        y_outer = int(center[1] + outer_radii[i] * np.sin(angle_rad))
        
        # Draw line from inner to outer edge
        cv2.line(overlay, (x_inner, y_inner), (x_outer, y_outer), (0, 255, 0), 1)
        
        # Mark points
        cv2.circle(overlay, (x_inner, y_inner), 2, (0, 0, 255), -1)
        cv2.circle(overlay, (x_outer, y_outer), 2, (255, 0, 0), -1)

cv2.circle(overlay, center, 5, (255, 255, 0), -1)
ax6.imshow(overlay)
ax6.set_title('Distance Vectors Overlay', fontweight='bold')
ax6.axis('off')

plt.tight_layout()
plt.savefig('output/comprehensive_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComprehensive analysis visualization created and saved!")

## 11. Generate Summary Report

Compile a summary dataset with distance values, statistical measures (mean, min, max, standard deviation), and export results as a table or CSV file.

In [ ]:
# Create DataFrame with all measurements
results_df = pd.DataFrame({
    'Angle (degrees)': bin_centers,
    'Inner Radius (pixels)': inner_radii,
    'Outer Radius (pixels)': outer_radii,
    'Distance (pixels)': distances
})

# Display first few rows
print("Measurement Results (first 10 rows):")
print(results_df.head(10).to_string(index=False))
print("\n...")
print("\n" + results_df.tail(5).to_string(index=False))

# Save to CSV
Path('output').mkdir(exist_ok=True)
csv_path = 'output/cam_ring_measurements.csv'
results_df.to_csv(csv_path, index=False)
print(f"\nResults saved to: {csv_path}")

# Create summary statistics
summary_stats = {
    'Metric': [
        'Total angles analyzed',
        'Angular resolution (degrees)',
        'Image center (x, y)',
        'Minimal enclosing circle radius (pixels)',
        '',
        'Inner Radius - Mean',
        'Inner Radius - Min',
        'Inner Radius - Max',
        'Inner Radius - Std Dev',
        '',
        'Outer Radius - Mean',
        'Outer Radius - Min',
        'Outer Radius - Max',
        'Outer Radius - Std Dev',
        '',
        'Distance - Mean',
        'Distance - Min',
        'Distance - Max',
        'Distance - Std Dev',
        'Distance - Median',
        'Total Variation',
        'Coefficient of Variation (%)',
        '',
        'Min Distance At',
        'Max Distance At'
    ],
    'Value': [
        len(bin_centers),
        angle_resolution,
        f"({center[0]}, {center[1]})",
        f"{radius_circle:.2f}",
        '',
        f"{inner_radii.mean():.2f}",
        f"{inner_radii.min():.2f}",
        f"{inner_radii.max():.2f}",
        f"{inner_radii.std():.2f}",
        '',
        f"{outer_radii.mean():.2f}",
        f"{outer_radii.min():.2f}",
        f"{outer_radii.max():.2f}",
        f"{outer_radii.std():.2f}",
        '',
        f"{distances.mean():.2f}",
        f"{distances.min():.2f}",
        f"{distances.max():.2f}",
        f"{distances.std():.2f}",
        f"{np.median(distances):.2f}",
        f"{max_variation:.2f}",
        f"{cv:.2f}",
        '',
        f"{min_distance_angle:.1f}°",
        f"{max_distance_angle:.1f}°"
    ]
}

summary_df = pd.DataFrame(summary_stats)

# Display summary statistics
print("\n" + "="*60)
print("ANALYSIS SUMMARY STATISTICS")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

# Save summary to CSV
summary_csv_path = 'output/analysis_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nSummary statistics saved to: {summary_csv_path}")

# Create a visual summary table
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('tight')
ax.axis('off')

# Prepare table data
table_data = [
    ['Metric', 'Value'],
    ['Analysis Parameters', ''],
    ['Total Angles', f'{len(bin_centers)}'],
    ['Angular Resolution', f'{angle_resolution}°'],
    ['Ring Center', f'({center[0]}, {center[1]})'],
    ['', ''],
    ['Radial Statistics (pixels)', ''],
    ['Distance Mean', f'{distances.mean():.2f}'],
    ['Distance Min', f'{distances.min():.2f}'],
    ['Distance Max', f'{distances.max():.2f}'],
    ['Distance Std Dev', f'{distances.std():.2f}'],
    ['Distance Median', f'{np.median(distances):.2f}'],
    ['', ''],
    ['Inner Radius Mean', f'{inner_radii.mean():.2f}'],
    ['Outer Radius Mean', f'{outer_radii.mean():.2f}'],
    ['', ''],
    ['Variation Metrics', ''],
    ['Total Variation', f'{max_variation:.2f}'],
    ['Coeff. of Variation', f'{cv:.2f}%'],
    ['Min Distance Location', f'{min_distance_angle:.1f}°'],
    ['Max Distance Location', f'{max_distance_angle:.1f}°'],
]

table = ax.table(cellText=table_data, cellLoc='left', loc='center',
                colWidths=[0.5, 0.5])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style the header row
for i in range(2):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Style section headers
section_rows = [1, 6, 12, 16]
for row in section_rows:
    for col in range(2):
        table[(row, col)].set_facecolor('#E8F5E9')
        table[(row, col)].set_text_props(weight='bold')

plt.title('Cam Ring Analysis Summary Report', fontsize=14, fontweight='bold', pad=20)
plt.savefig('output/summary_table.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary report table visualization created!")
print("\nAll results have been generated and saved to the 'output' folder.")